# Cohort Analysis

## Objective

Analyze **monthly customer retention after acquisition** by grouping customers according to the month of their first purchase and tracking the percentage of each cohort that remains active in later months.

### Key definitions

- **Cohort:** A group of customers who made their first-ever purchase in the same month.
- **Cohort Month:** The customer's first-purchase month.
- **Cohort Index:** The number of calendar months between the cohort month and a particular purchase month.
- **M0:** Acquisition month. Retention is 100% by definition.
- **M1–M12:** Monthly customer activity from one to twelve calendar months after acquisition.
- **Retention Rate:** Unique active customers in a cohort month-index ÷ original cohort size.

> A customer is counted only once within a cohort-index even if they place multiple orders during that month.


## 1. Import Libraries


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


## 2. Load Orders Data

The complete orders dataset is loaded first. Only the fields required for cohort analysis are then copied into a separate working DataFrame.


In [ ]:
df = pd.read_csv(
    "../../../02_Database/Dataset/10_Orders.csv"
)

ot = df[['order_id', 'customer_id', 'order_datetime']].copy()

ot.head()


## 3. Data Validation

Before creating cohorts, validate the fields that directly affect customer and time-based calculations.

Checks performed:

- Missing customer IDs
- Missing order dates
- Duplicate order IDs
- Dataset date coverage
- Datatypes and dimensions


In [ ]:
ot['order_datetime'] = pd.to_datetime(ot['order_datetime'])

validation_summary = pd.Series({
    'Rows': ot.shape[0],
    'Columns': ot.shape[1],
    'Duplicate Order IDs': ot['order_id'].duplicated().sum(),
    'Missing Customer IDs': ot['customer_id'].isna().sum(),
    'Missing Order Dates': ot['order_datetime'].isna().sum(),
    'Earliest Order': ot['order_datetime'].min(),
    'Latest Order': ot['order_datetime'].max()
})

validation_summary


## 4. Create Order Month

Cohort analysis is performed at **monthly granularity**, so each order timestamp is converted to a monthly Period.

Example: `2025-03-18 14:30:00` → `2025-03`


In [ ]:
ot['order_month'] = ot['order_datetime'].dt.to_period('M')

ot.head()


## 5. Assign Cohort Month

Each customer is assigned to the month of their **first-ever purchase**.

`transform('min')` is used because it calculates each customer's earliest order month and broadcasts that value back to every order row belonging to the same customer.


In [ ]:
ot['cohort_month'] = (
    ot.groupby('customer_id')['order_month']
      .transform('min')
)

ot.head()


## 6. Calculate Cohort Index

The cohort index measures how many calendar months after acquisition a purchase occurred.

For example, if a customer's cohort month is `2025-01`:

- Purchase in `2025-01` → M0
- Purchase in `2025-02` → M1
- Purchase in `2025-03` → M2

This is **monthly activity retention**, not an order-sequence measure. A customer can skip one month and become active again in a later month.


In [ ]:
ot['cohort_index'] = (
    (ot['order_month'].dt.year - ot['cohort_month'].dt.year) * 12
    + (ot['order_month'].dt.month - ot['cohort_month'].dt.month)
)

ot.head()


## 7. Count Unique Active Customers

For every combination of cohort month and cohort index, count the number of **distinct customers** who purchased.

`nunique()` is necessary because one customer may place multiple orders in the same month but should count as only one active customer for retention analysis.


In [ ]:
month_wise_cohort_count = (
    ot.groupby(['cohort_month', 'cohort_index'])['customer_id']
      .nunique()
)

month_wise_cohort_count.head(10)


## 8. Build the Cohort Customer Matrix

The grouped Series is reshaped so that:

- Rows = acquisition cohorts
- Columns = cohort indices
- Values = unique active customers

### Reporting window

The final analysis uses **36 fully matured monthly cohorts** and follows each one through **M0–M12**.

The orders dataset currently ends on **25 July 2026**. Because July 2026 is only a partial month, the latest cohort with twelve **fully completed** post-acquisition months is **June 2025**.

Therefore, the reporting period is:

**July 2022 to June 2025** → 36 cohorts

For these mature cohorts, a missing cohort-index value means no customer was active in that observed month, so it is correctly replaced with `0`.


In [ ]:
cohort_customer_matrix_all = month_wise_cohort_count.unstack()

cohort_customer_matrix = (
    cohort_customer_matrix_all
    .loc['2022-07':'2025-06', 0:12]
    .fillna(0)
)

cohort_customer_matrix.columns = [
    'M0', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6',
    'M7', 'M8', 'M9', 'M10', 'M11', 'M12'
]

cohort_customer_matrix


## 9. Calculate Cohort Retention Rates

Each row is divided by its own M0 customer count.

This converts the customer-count matrix into a retention-percentage matrix:

**Retention at Mx = Active customers at Mx ÷ Customers acquired at M0 × 100**


In [ ]:
cohort_retention_matrix = (
    cohort_customer_matrix
    .div(cohort_customer_matrix['M0'], axis=0)
    .mul(100)
    .round(2)
)

cohort_retention_matrix


## 10. Validate the Retention Matrix

Every mature cohort must have **M0 = 100%** because M0 is the acquisition cohort compared with itself.


In [ ]:
m0_validation = cohort_retention_matrix['M0'].eq(100).all()

print("All M0 retention values equal 100%:", m0_validation)
print("Retention matrix shape:", cohort_retention_matrix.shape)


## 11. Average Retention by Months Since Acquisition

To summarize the overall retention pattern, calculate the mean retention rate across the 36 acquisition cohorts for each cohort index.

M0 is retained in the table for reference but excluded from the trend chart because its fixed 100% value would compress the much smaller post-acquisition retention rates.


In [ ]:
average_retention = cohort_retention_matrix.mean(axis=0).round(2)

average_retention


## 12. Visualization — Average Post-Acquisition Retention

The line chart focuses on **M1–M12**.

It answers:

> On average, what percentage of customers from an acquisition cohort are active in each later calendar month?

### Important interpretation note

This is **monthly activity retention**. A customer does not need to purchase every month to appear in a later cohort index. For example, a customer may be inactive in M2 and purchase again in M3.


In [ ]:
post_acquisition_retention = average_retention.iloc[1:]

plt.figure(figsize=(10, 5))
plt.plot(
    post_acquisition_retention.index,
    post_acquisition_retention.values,
    marker='o'
)

plt.xlabel("Months Since Acquisition")
plt.ylabel("Average Retention Rate (%)")
plt.title("Average Monthly Customer Retention After Acquisition")
plt.grid(axis='y', alpha=0.3)
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()


## 13. Business Insights

Use the retention matrix and the M1–M12 trend to document only findings supported by the results.

Suggested questions:

- How much customer activity remains one month after acquisition?
- Does retention generally decline, stabilize, or fluctuate across M1–M12?
- Are there later-month recoveries that suggest customers return after skipping months?
- Is the observed pattern reasonable for a consumer-electronics retailer with naturally longer repurchase cycles?
- Are any individual acquisition cohorts materially stronger or weaker than the overall pattern?

> **Business insights to be finalized after reviewing the executed output.**


## 14. Analysis Notes & Limitations

- Cohorts are defined by **first-purchase month**, not registration month.
- Retention measures whether a customer was active in a specific later calendar month; it does not measure continuous survival.
- Multiple purchases by the same customer in one month count as one active customer.
- Only mature cohorts with a complete M0–M12 observation window are used for the final comparison.
- The dataset ends on 25 July 2026, so July 2026 is treated as a partial observation month. Therefore, June 2025 is the latest acquisition cohort included, ensuring that every selected cohort has a complete M0–M12 observation window.
- This analysis measures purchase activity only; it does not explain *why* customers return or fail to return.


## Conclusion

This cohort analysis provides a consistent view of customer purchasing activity during the first twelve months after acquisition. The retention matrix preserves cohort-level detail, while the average retention curve summarizes the overall post-acquisition pattern.

The next step is to convert the observed patterns into business insights and recommendations within the broader Customer Analysis engagement.
